In [2]:
# conda activate dream

library(sva)
library(edgeR)
library(limma)
library(dplyr)
library(data.table)

setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

In [3]:
batch_ANOVA <- function(pca, sampleinfo, batch_cols) {
    for (i in batch_cols) {
        batch_label <- colnames(sampleinfo)[i]
        batch <- factor(sampleinfo[,i])
        if (length(levels(batch)) > 1) {
            if (batch_label == "Mean_age") {
                batch <- sampleinfo[,i] 
            }
            fit <- lm(pca$x[,2] ~ batch)
            pc_test <- anova(fit)$`Pr(>F)`[1]
            print(paste(batch_label, p.adjust(pc_test, "BH")))
        }
    }
}

In [4]:
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv", data.table=FALSE)

In [14]:
sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))

batch_cols <- c(
    grep("Mean_age", colnames(sampleinfo)),
    grep("SMCENTER", colnames(sampleinfo)),
    grep("SMNABTCH$", colnames(sampleinfo)), 
    grep("SMGEBTCH$", colnames(sampleinfo)),
    grep("SEX", colnames(sampleinfo)),
    grep("SMTSD", colnames(sampleinfo)),
    grep("DTHHRDY", colnames(sampleinfo))
)

## Starting from TPM SN QN rd2 data

### Prep data

In [ ]:
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv", data.table=FALSE)
expr <- fread("GTEx_cortex_TPM_rd2_SampleNetworks/All_02-16-53/GTEx_cortex_TPM_rd2_All_501_outliers_removed.csv", data.table=FALSE)

In [8]:
expr[1:4,1:4]

,datExprT...c.1.skip1..,GTEX.111FC.0011.R3b.SM.GJ3PN,GTEX.117XS.0011.R3a.SM.GIN8W,GTEX.1192X.0011.R3b.SM.GIN8Y
,<chr>,<dbl>,<dbl>,<dbl>
1,DDX11L1,0.0000000,0.000000,0.000000
2,WASH7P,2.5457173,3.642921,2.463347
3,MIR6859-1,0.0000000,0.000000,0.000000
4,MIR1302-2HG,0.0477403,0.000000,0.000000


In [9]:
sampleinfo[,1] <- make.names(sampleinfo[,1])
sampleinfo <- sampleinfo[match(colnames(expr)[-1], sampleinfo[,1]),]
sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))

In [10]:
# For duplicate genes, choose row with highest mean expression

mean_expr <- data.frame(
    Index=1:nrow(expr), 
    Gene=expr[,1], 
    Expr=rowMeans(expr[,-1])
)

mean_expr <- mean_expr %>%
    group_by(Gene) %>%
    slice_max(Expr)

print(dim(mean_expr))

expr <- expr[mean_expr$Index,]

# Subset to genes in the top X percentile of mean expression

prob <- .6
mean_expr <- rowMeans(expr[,-1])
print(sum(mean_expr >= quantile(mean_expr, prob)))
expr_filtered <- expr[mean_expr >= quantile(mean_expr, prob),]

[1] 55692     3
[1] 22277


In [11]:
rownames(expr_filtered) <- expr_filtered[,1]
dat <- expr_filtered[,-1]
dim(dat)

[1] 22277   501

### Check significance of batch variables

In [12]:
# Before ComBat

pca <- prcomp(t(dat), center=TRUE, scale.=TRUE)

In [15]:
batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 0.015510282443709"
[1] "SMCENTER 8.91598908338723e-87"
[1] "SMNABTCH 8.52924667374512e-26"
[1] "SMGEBTCH 1.22443325388007e-11"
[1] "SEX 0.153850676386419"
[1] "SMTSD 2.61126263815865e-90"
[1] "DTHHRDY 0.214553918008083"


### Run ComBat

In [17]:
mod <- model.matrix(~Mean_age + as.factor(SEX) + as.factor(SMTSD), data=sampleinfo)

batch <- factor(sampleinfo$SMGEBTCH)

expr_combat1 <- sva::ComBat(
    dat,
    batch,
    mod=mod,
    mean.only=FALSE,
    par.prior=TRUE,
    prior.plots=FALSE
)

Found 1317 genes with uniform expression within a single batch (all zeros); these will not be adjusted for batch.


Using the 'mean only' version of ComBat

Found173batches

Note: one batch has only one sample, setting mean.only=TRUE

Adjusting for4covariate(s) or covariate level(s)

Standardizing Data across genes

Fitting L/S model and finding priors

Finding parametric adjustments

Adjusting the Data




### Check significance of batch variables

In [18]:
# After ComBat

pca <- prcomp(t(expr_combat1), center=TRUE, scale.=TRUE)

batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 0.0938632610218132"
[1] "SMCENTER 1.96363192409567e-129"
[1] "SMNABTCH 1.25040920917601e-32"
[1] "SMGEBTCH 3.03841356887261e-06"
[1] "SEX 0.095489325034402"
[1] "SMTSD 3.68645233998238e-137"
[1] "DTHHRDY 0.348631538035072"


### Run ComBat again

In [19]:
mod <- model.matrix(~Mean_age + as.factor(SEX) + as.factor(sampleinfo$SMTSD), data=sampleinfo)

batch <- as.factor(sampleinfo$SMGEBTCH)

expr_combat2 <- sva::ComBat(
    expr_combat1,
    batch=batch,
    mod=mod,
    mean.only=FALSE,
    par.prior=TRUE,
    prior.plots=FALSE
)

Found 1317 genes with uniform expression within a single batch (all zeros); these will not be adjusted for batch.


Using the 'mean only' version of ComBat

Found173batches

Note: one batch has only one sample, setting mean.only=TRUE

Adjusting for4covariate(s) or covariate level(s)

Standardizing Data across genes

Fitting L/S model and finding priors

Finding parametric adjustments

Adjusting the Data




### Check significance of batch variables

In [20]:
pca <- prcomp(t(expr_combat2), center=TRUE, scale.=TRUE)

batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 0.13004596907701"
[1] "SMCENTER 1.71452051851891e-136"
[1] "SMNABTCH 1.02562465410297e-33"
[1] "SMGEBTCH 1.24831697268674e-05"
[1] "SEX 0.0903052859833676"
[1] "SMTSD 1.17942841538618e-145"
[1] "DTHHRDY 0.416928297792487"


In [ ]:
# Didn't improve batch significance that much. Saving less processed version.

In [21]:
fwrite(expr_combat1, file=paste0("data/GTEx_cortex_TPM_rd2_All_501_outliers_removed_ComBat_SMGEBTCH_corrected.csv"), row.names=TRUE)

x being coerced from class: matrix to data.table



## Starting from counts SN rd2 data

### Prep data

In [22]:
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv", data.table=FALSE)
expr <- fread("GTEx_cortex_counts_rd2_SampleNetworks/All_02-39-09/GTEx_cortex_counts_rd2_All_501_outliers_removed.csv", data.table=FALSE)

In [25]:
expr[1:5,1:4]

,datExprT...c.1.skip1..,GTEX.111FC.0011.R3b.SM.GJ3PN,GTEX.117XS.0011.R3a.SM.GIN8W,GTEX.1192X.0011.R3b.SM.GIN8Y
,<chr>,<int>,<int>,<int>
1,DDX11L1,0,0,0
2,WASH7P,144,164,134
3,MIR6859-1,0,0,0
4,MIR1302-2HG,2,0,0
5,FAM138A,0,0,0


In [26]:
sampleinfo[,1] <- make.names(sampleinfo[,1])
sampleinfo <- sampleinfo[match(colnames(expr)[-1], sampleinfo[,1]),]
sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))

In [27]:
# For duplicate genes, choose row with highest mean expression

mean_expr <- data.frame(
    Index=1:nrow(expr), 
    Gene=expr[,1], 
    Expr=rowMeans(expr[,-1])
)

mean_expr <- mean_expr %>%
    group_by(Gene) %>%
    slice_max(Expr)

print(dim(mean_expr))

expr <- expr[mean_expr$Index,]

# Subset to genes in the top X percentile of mean expression

prob <- .6
mean_expr <- rowMeans(expr[,-1])
print(sum(mean_expr >= quantile(mean_expr, prob)))
expr_filtered <- expr[mean_expr >= quantile(mean_expr, prob),]

[1] 55442     3
[1] 22178


In [28]:
dim(expr_filtered)

[1] 22178   502

In [29]:
total_expr <- colSums(expr_filtered[,-1])
expr_norm <- sweep(expr_filtered[,-1], MARGIN=2, FUN="/", STATS=total_expr) * 1e4

In [30]:
rownames(expr_norm) <- expr_filtered[,1]
dat <- expr_norm
dim(dat)

[1] 22178   501

### Check significance of batch variables

In [31]:
# Before ComBat

pca <- prcomp(t(dat), center=TRUE, scale.=TRUE)

In [ ]:
batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 0.00344516235382899"
[1] "SMCENTER 1.66702982593877e-24"
[1] "SMNABTCH 1.9630378682996e-07"
[1] "SMGEBTCH 0.00408343915285182"
[1] "SEX 0.000361061746848544"
[1] "SMTSD 2.50638132475145e-26"
[1] "DTHHRDY 2.46286897851178e-10"


### Run ComBat

In [32]:
mod <- model.matrix(~Mean_age + as.factor(SEX) + as.factor(SMTSD), data=sampleinfo)

batch <- factor(sampleinfo$SMGEBTCH)

expr_combat1 <- sva::ComBat(
    dat,
    batch,
    mod=mod,
    mean.only=FALSE,
    par.prior=TRUE,
    prior.plots=FALSE
)

Found 268 genes with uniform expression within a single batch (all zeros); these will not be adjusted for batch.


Using the 'mean only' version of ComBat

Found177batches

Note: one batch has only one sample, setting mean.only=TRUE

Adjusting for4covariate(s) or covariate level(s)

Standardizing Data across genes

Fitting L/S model and finding priors

Finding parametric adjustments

Adjusting the Data




### Check significance of batch variables

In [33]:
# After ComBat

pca <- prcomp(t(expr_combat1), center=TRUE, scale.=TRUE)

batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 1.43463633975612e-05"
[1] "SMCENTER 8.21544122792259e-72"
[1] "SMNABTCH 1.23780179343228e-14"
[1] "SMGEBTCH 0.418394612274922"
[1] "SEX 0.0146091363796296"
[1] "SMTSD 1.33189406493509e-74"
[1] "DTHHRDY 4.20625993290282e-07"


In [34]:
fwrite(expr_combat1, file=paste0("data/GTEx_cortex_counts_rd2_All_501_outliers_removed_ComBat_SMGEBTCH_corrected.csv"), row.names=TRUE)

x being coerced from class: matrix to data.table



In [35]:
expr_combat1

,GTEX.111FC.0011.R3b.SM.GJ3PN,GTEX.117XS.0011.R3a.SM.GIN8W,GTEX.1192X.0011.R3b.SM.GIN8Y,GTEX.11DXW.0011.R3b.SM.DNZZE,GTEX.11GS4.0011.R3b.SM.GJ3RI,GTEX.11GSO.0011.R3b.SM.57WB2,GTEX.11GSP.0011.R3a.SM.9QEGF,GTEX.11NUK.0011.R3b.SM.GJ3RO,GTEX.11NV4.0011.R3b.SM.GINAJ,GTEX.11O72.0011.R3a.SM.H65ZL,⋯,GTEX.XLM4.0011.R10A.SM.4AT5P,GTEX.Y8DK.0011.R10A.SM.4SOK1,GTEX.YJ89.0011.R10a.SM.4SOK9,GTEX.ZF28.0011.R10a.SM.4WWEH,GTEX.ZUA1.0011.R10a.SM.51MT6,GTEX.ZV68.0011.R10a.SM.51MT7,GTEX.ZVT3.0011.R10b.SM.57WB6,GTEX.ZVZQ.0011.R10b.SM.51MRT,GTEX.ZXG5.0011.R10a.SM.57WDD,GTEX.ZZPT.0011.R10b.SM.GPI8B
A1BG,0.046613223,4.188489e-02,0.0440754982,0.062893963,0.047815130,0.037267769,0.0225745085,0.053463744,0.033373276,0.0421534071,⋯,0.046823775,0.039194776,0.0354692161,0.033885795,0.040281327,0.0301154289,0.062227448,0.0485875566,0.0376822590,0.045691495
A1BG-AS1,0.038178968,1.148113e-02,0.0180885248,0.017847235,0.023334021,0.027652678,0.0049391977,0.016094960,0.013411110,0.0089965235,⋯,0.016597189,0.017531558,0.0230632413,0.019854289,0.016793511,0.0066266322,0.020417720,0.0288381847,0.0124991893,0.014777802
A2M,0.874489282,1.424826e+00,1.7915915142,0.746703208,1.238553992,1.134626028,0.5811734417,0.955865060,1.328448682,1.6285538562,⋯,1.568130481,1.559369011,0.5517447319,1.027697087,1.979849316,1.6330933832,1.220331830,1.4038216517,0.8096837647,0.836316767
A2M-AS1,0.012091020,2.442174e-02,0.0216687250,0.022855020,0.021212277,0.021614851,0.0220160579,0.016608692,0.023200338,0.0112071203,⋯,0.025549823,0.022783923,0.0253998296,0.026061133,0.040508177,0.0179385315,0.022310270,0.0326889801,0.0172326009,0.005261059
A2ML1,0.025997187,3.587366e-02,0.1754240198,0.066599797,0.080950591,0.055747440,0.0998235693,0.031837626,0.024024676,0.0713867357,⋯,0.049662964,0.030593333,0.0385955220,0.046730755,0.048083263,0.0628080052,0.038934532,0.0391228996,0.0583399269,0.050246505
A2MP1,0.001333547,2.160876e-03,0.0013885233,0.002490928,0.001751794,0.002416213,0.0014678241,0.001061455,0.003056557,0.0016148609,⋯,0.003051612,0.001510759,0.0019991140,0.002473091,0.001570075,0.0018685765,0.002446224,0.0027201040,0.0002825508,0.001390426
A4GALT,0.021452925,3.492755e-02,0.0440882334,0.030082108,0.034905528,0.044716390,0.0398767973,0.018269464,0.015129611,0.0801217718,⋯,0.043187480,0.037312470,0.0340878450,0.046276778,0.050054098,0.0676693652,0.035228844,0.0338805946,0.0323850661,0.118607497
AAAS,0.253598871,2.366756e-01,0.2307928940,0.290244494,0.302122441,0.260870185,0.2422870356,0.261843117,0.220943717,0.2723154149,⋯,0.292685309,0.292812709,0.2532554911,0.295500647,0.262615650,0.2870674359,0.249809961,0.2918637640,0.2104218930,0.261573493
AACS,0.951621989,4.168160e-01,0.3744825465,0.578171125,0.543457677,0.871233188,0.1642607437,0.526594072,0.450316833,0.3976728087,⋯,0.559504486,0.650464916,0.5836274343,0.630552895,0.510966918,0.2610430001,0.478101141,0.7349791607,0.3306076290,0.519650363
AACSP1,0.006242163,3.619118e-03,0.0006287738,0.003325682,0.002115720,0.003309173,0.0007548658,0.003331557,0.001191879,0.0005850484,⋯,0.002161376,0.002692578,0.0010443738,0.002020176,0.003406260,0.0009672656,0.002951279,0.0043528316,0.0003548713,0.002703490
